# Session 4 · Lab B – Saving data and models: CSV vs. pickle vs. Parquet

**Runs locally** in VS Code with your `.venv` kernel (Lab A). Needs `data/yellow_tripdata_2024-01.parquet`
(`python get_data.py --months 2`).

Sooner or later you save a DataFrame or a trained model to disk. The file format decides
- how **fast** you can write and read it,
- how much **disk** it uses,
- whether the **data types** survive the round trip,
- **who** can open it (Excel? Spark? only Python?),
- and whether opening it is **safe**.

We measure all of this ourselves, on real taxi data – **your numbers will differ from your neighbour's**.
That is part of the lesson: always benchmark on your own data and hardware.

In [ ]:
import os, io, time, pickle, platform
from pathlib import Path
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq
import matplotlib.pyplot as plt

DATA = Path("../data")
OUT = DATA / "lab_b"          # scratch folder for the files we write
OUT.mkdir(exist_ok=True)

print("Python ", platform.python_version(), "| pandas", pd.__version__, "| pyarrow", pa.__version__)

In [ ]:
def mb(path):
    return os.path.getsize(path) / 1e6

def bench(fn, repeat=3):
    # best of `repeat` runs in seconds (the minimum is the least noisy estimate)
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter(); fn(); times.append(time.perf_counter() - t0)
    return min(times)

results = []   # we collect one row per format here

## 1. The data: one million real taxi trips
We take a random sample of 1 million trips from January 2024 (the full month has ~3 million).
If your laptop is slow, set `N_ROWS = 300_000`.

In [ ]:
N_ROWS = 1_000_000

raw = pd.read_parquet(DATA / "yellow_tripdata_2024-01.parquet")
df = raw.sample(n=min(N_ROWS, len(raw)), random_state=42).reset_index(drop=True)
del raw
df.info(memory_usage="deep")

## 2. CSV – the default everybody knows

In [ ]:
csv_path = OUT / "trips.csv"
t_write = bench(lambda: df.to_csv(csv_path), repeat=1)    # writing is slow -> only once
t_read  = bench(lambda: pd.read_csv(csv_path))
print(f"CSV: write {t_write:.2f} s | read {t_read:.2f} s | {mb(csv_path):.0f} MB")

back = pd.read_csv(csv_path)
back.head(3)

**Look closely at the result:**
1. There is a new column `Unnamed: 0`. Where does it come from? (Hint: `to_csv(..., index=False)`.)
2. Compare the dtypes before and after:

In [ ]:
pd.DataFrame({"original": df.dtypes.astype(str), "after CSV": back.dtypes.astype(str)}).dropna()

The timestamps came back as **text**. CSV stores only characters – no types, no schema.
pandas has to *guess* the types again on every read, and it cannot guess timestamps or categories.

**Your turn (5 min):** read the CSV back so that the two timestamp columns are real datetimes and
the index column disappears. Then think: would you want to write this for 50 columns?

In [ ]:
# TODO: use the parameters index_col=... and parse_dates=[...] of pd.read_csv
back = pd.read_csv(csv_path)
back.dtypes.head(4)

## 3. Smaller dtypes – and who remembers them?
Many columns use far bigger types than needed: `VendorID` is 1 or 2 but stored as a 32-bit integer,
`store_and_fwd_flag` is `"Y"`/`"N"` but stored as a string per row.

In [ ]:
def set_dtypes(d):
    d = d.copy()
    for c in ["VendorID", "payment_type"]:
        d[c] = d[c].astype("int8")
    for c in ["passenger_count", "RatecodeID"]:
        d[c] = d[c].astype("Int8")                  # capital I = nullable integer (has missing values)
    for c in ["PULocationID", "DOLocationID"]:
        d[c] = d[c].astype("int16")
    d["store_and_fwd_flag"] = d["store_and_fwd_flag"].astype("category")
    money = d.select_dtypes("float64").columns
    d[money] = d[money].astype("float32")          # fine for analytics, NOT for accounting
    return d

before = df.memory_usage(deep=True).sum() / 1e6
df = set_dtypes(df)
after = df.memory_usage(deep=True).sum() / 1e6
print(f"in memory: {before:.0f} MB -> {after:.0f} MB")

We spent effort on these dtypes. **A good file format should keep them.**
From now on, every format gets the same test: write, read, size – and *do the dtypes survive?*

In [ ]:
def run_format(name, path, write, read, repeat=3):
    t_w = bench(write, repeat=1)
    t_r = bench(read, repeat=repeat)
    same = read().dtypes.astype(str).equals(df.dtypes.astype(str))
    results.append({"format": name, "write_s": t_w, "read_s": t_r, "MB": mb(path), "dtypes_kept": same})
    print(f"{name:<16} write {t_w:6.2f} s | read {t_r:6.2f} s | {mb(path):6.0f} MB | dtypes kept: {same}")

p = OUT / "trips.csv"
run_format("CSV", p, lambda: df.to_csv(p, index=False), lambda: pd.read_csv(p))

## 4. pickle – Python's own "freeze-dry"
`pickle` turns **any** Python object into bytes and back. `df.to_pickle()` is a thin wrapper around it.

In [ ]:
p = OUT / "trips.pkl"
run_format("pickle", p, lambda: df.to_pickle(p), lambda: pd.read_pickle(p))

## 5. Parquet – columnar, compressed, typed
You know Parquet from Session 1. Two compression codecs: `snappy` (default, fast) and `zstd` (smaller).

In [ ]:
p = OUT / "trips_snappy.parquet"
run_format("Parquet snappy", p, lambda: df.to_parquet(p, compression="snappy"), lambda: pd.read_parquet(p))

p = OUT / "trips_zstd.parquet"
run_format("Parquet zstd", p, lambda: df.to_parquet(p, compression="zstd"), lambda: pd.read_parquet(p))

p = OUT / "trips.feather"
run_format("Feather (Arrow)", p, lambda: df.to_feather(p), lambda: pd.read_feather(p))

In [ ]:
res = pd.DataFrame(results).set_index("format")
display(res.round(2))

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
for a, col in zip(ax, ["write_s", "read_s", "MB"]):
    res[col].plot.barh(ax=a, title=col); a.invert_yaxis()
plt.tight_layout()

## 6. Reading only what you need
In practice you rarely need all 19 columns. Parquet stores each column separately (+ min/max statistics),
so it can skip the rest. pickle is one opaque blob: **all or nothing**.

**Your turn (5 min):** read only `tpep_pickup_datetime`, `trip_distance` and `tip_amount`,
and only credit-card trips (`payment_type == 1`), directly from the Parquet file.
Compare **time and memory** with loading the whole pickle and filtering afterwards.

In [ ]:
pq_path, pkl_path = OUT / "trips_zstd.parquet", OUT / "trips.pkl"
cols = ["tpep_pickup_datetime", "trip_distance", "tip_amount"]

# TODO 1: time pd.read_parquet(...) with the parameters columns=... and filters=[(...)]
# TODO 2: time pd.read_pickle(...) followed by a pandas filter on payment_type == 1

**Surprised?** pickle can be *as fast or faster* here: it is close to a raw memory dump, and the whole file fits in RAM.
The real difference is **what has to fit into memory**: pickle always loads every row and column first.
With 50 GB of trips the pickle does not load at all, while Parquet still reads just the 3 columns and the
row groups you need. Speed on small data is not the same as scalability.

### What should you use for *data*?
| | CSV | pickle | Parquet |
|---|---|---|---|
| Human-readable / Excel | ✓ | ✗ | ✗ |
| Keeps dtypes | ✗ | ✓ | ✓ |
| Compressed | ✗ | ✗ (by default) | ✓ |
| Read single columns | ✗ | ✗ | ✓ |
| Spark, DuckDB, Polars, R can read it | ✓ | ✗ (Python only) | ✓ |
| Safe to open from strangers | ✓ | **✗ (see section 8)** | ✓ |

**Data** → Parquet. **CSV** → only to hand over to humans/Excel. **pickle** → not for data you share or keep.
So why does pickle exist at all? Because some things are *not tables*.

## 7. Where pickle shines: Python objects such as trained models
A trained scikit-learn model is a Python object with fitted parameters inside – no table format can store it.
We train a small model that predicts the **trip duration in minutes** (the same task we will scale up with Spark later).

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import sklearn

d = df.copy()
d["duration_min"] = (d.tpep_dropoff_datetime - d.tpep_pickup_datetime).dt.total_seconds() / 60
d = d[(d.duration_min.between(1, 180)) & (d.trip_distance.between(0.1, 60))]
d["hour"] = d.tpep_pickup_datetime.dt.hour
d["weekday"] = d.tpep_pickup_datetime.dt.weekday

FEATURES = ["trip_distance", "hour", "weekday", "PULocationID", "DOLocationID"]
sample = d.sample(n=min(200_000, len(d)), random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(sample[FEATURES], sample.duration_min, test_size=0.2, random_state=1)

model = HistGradientBoostingRegressor(max_iter=100, random_state=1).fit(X_tr, y_tr)
print(f"MAE on test: {mean_absolute_error(y_te, model.predict(X_te)):.2f} minutes")

In [ ]:
import joblib

with open(OUT / "model.pkl", "wb") as f:          # plain pickle
    pickle.dump(model, f)
joblib.dump(model, OUT / "model.joblib")            # joblib: pickle + efficient for big numpy arrays

with open(OUT / "model.pkl", "rb") as f:
    loaded = pickle.load(f)

print("identical predictions:", np.allclose(loaded.predict(X_te), model.predict(X_te)))
print(f"model.pkl {mb(OUT/'model.pkl'):.2f} MB | model.joblib {mb(OUT/'model.joblib'):.2f} MB")

**Your turn (3 min):** `joblib.dump` has a `compress=` parameter. Save the model with `compress=3`.
How much smaller is the file, and does loading still give identical predictions?

In [ ]:
# TODO: joblib.dump(..., compress=3), then joblib.load and compare predictions

### A pickle only works together with its environment
A pickled model stores *references* to classes (`sklearn.ensemble...HistGradientBoostingRegressor`), not their code.
Load it with a different scikit-learn version and you may get a warning, wrong behaviour or an error.
This is exactly why Lab A matters. Good practice: **save the versions next to the model.**

In [ ]:
bundle = {
    "model": model,
    "features": FEATURES,
    "sklearn_version": sklearn.__version__,
    "python_version": platform.python_version(),
    "trained_on": "yellow_tripdata_2024-01, 200k sample",
}
joblib.dump(bundle, OUT / "model_bundle.joblib")

b = joblib.load(OUT / "model_bundle.joblib")
if b["sklearn_version"] != sklearn.__version__:
    print("⚠️  model was trained with scikit-learn", b["sklearn_version"])
print(b["features"], "| trained with sklearn", b["sklearn_version"])

## 8. ⚠️ Never unpickle files you don't trust
Unpickling does not just *read* data – it may **run code**. An object can tell pickle
"to rebuild me, call this function". The demo below is harmless: the function is `print`.
An attacker would use something like `os.system` instead.

In [ ]:
class Surprise:
    def __reduce__(self):                      # tells pickle how to "rebuild" the object
        return (print, ("😱 This line ran while LOADING the file – nobody called a method!",))

payload = pickle.dumps(Surprise())
print(f"payload is just {len(payload)} bytes ... now we load it:")
pickle.loads(payload)

The same applies to `pd.read_pickle`, `joblib.load` and older model files from the internet.
Rules of thumb:
- Only load pickles **you created yourself** or that come from a source you fully trust.
- To *share* data, use Parquet. To share models across languages/tools there are formats such as ONNX
  (and Spark has its own `model.save()` – you will see it in Lab D).

Clean up the scratch files when you're done:

In [ ]:
import shutil
shutil.rmtree(OUT)

## 9. Exam-style questions
1. A colleague stores the monthly taxi extract as CSV "because everybody can open it". Give **three**
   concrete disadvantages you observed in this lab, each with the measured effect on your machine.
2. Your pandas DataFrame has a `category` column and nullable integers. Which of CSV, pickle and Parquet
   keep these types after a round trip? Explain why CSV cannot.
3. On small data, loading a pickle can be as fast as reading Parquet. Why does Parquet still scale better when you
   only need 3 of 19 columns of a 50 GB dataset? Use the words *columnar* and *row group statistics*.
4. Scenario: you trained a scikit-learn model on your laptop; the model will be used by a batch job on a server
   in six months. Which format do you save it in, and which **two** pieces of information do you store with it?
5. A student downloads `best_model.pkl` from a forum and runs `pickle.load`. Explain the risk in two sentences.
6. True or false, with justification: "pickle is the best format for large datasets because it is fastest to read."